# DCT Laboratory — Volume I, Chapter 6
## Enterprise Transformation Operators
**Seed `26106`** · Companion to Chapter 6 and **AXIOM Lab 1.6** (module **AXIOM-06**) · Mirrored in `DCT_V1_Ch06_Lab.xlsx`

This laboratory reproduces **Worked Example 6.1, *Strategy or Trade? Meridian's Automation Playbook***, with the
book's declarations, and works the five steps of AXIOM Lab 1.6, *Sequencing Meridian's Moves*: forge the operators,
compose programs, test the playbook, explore reachability, stress the execution. Every Meridian number below is the
book's (Worked Example 6.1 and Table 6.4, Figures 6.1–6.4, Section 6.5, Online Appendix 6.B); the final cell checks
them. Money is in \$ million.

## The declarations (Worked Example 6.1, Step 1; Figure 6.3)

Meridian today (Table 3.1): liquidity \$620 million, technology platform 41, workforce capability 58, operational
efficiency 71. The board's floor is \$350 million of liquidity (Table 5.4). The lab's repertoire has four moves, each
declared with its effect and its domain, the set of states from which it can be taken (Definitions 6.1 and 6.3):

- **S**, sell part of Industrial Systems as designed, the shared engineering team going with the unit (Worked
  Example 3.1): \$112 million net, the unit takes 10 percent of the technology position, and workforce capability
  falls by 5 points. It needs the unit unsold.
- **S\***, the same sale keeping the engineers: \$92 million, and capability is kept. It needs the unit unsold.
- **D**, the digital build: commits half of the liquidity and adds 0.05 technology points per \$1 million invested.
  It needs workforce capability of at least 55 and must leave the floor intact, so it needs \$700 million first.
- **A**, the automation round: costs \$120 million, closes half of the gap between efficiency and the plants'
  ceiling of 85, and retrains staff, adding a point of workforce capability. It needs liquidity of at least
  \$470 million, so that the floor holds after the round, and workforce capability of at least 55. Every efficiency
  point gained is worth \$10 million a year in lower costs.

In [ ]:
import math
from collections import namedtuple
from decimal import Decimal, ROUND_HALF_UP
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
SEED = 26106

# Meridian today (Table 3.1) and the board's floor (Table 5.4); money in $ million
State = namedtuple("State", "L T W E sold")    # liquidity, technology, workforce capability, efficiency, unit sold
TODAY = State(L=620.0, T=41.0, W=58.0, E=71.0, sold=False)
FLOOR = 350.0

# The automation round (Worked Example 6.1)
COST, CEIL, CLOSE = 120.0, 85.0, 0.5        # cost; the plants' technical ceiling; share of the gap closed
W_MIN, W_GAIN, VALUE = 55.0, 1.0, 10.0      # capability to staff the lines; retraining; $ a year per point gained
PRE_A = FLOOR + COST                        # 470: the floor still holds after the round is paid for
# The sale of part of Industrial Systems and the digital build (Worked Example 3.1)
PROCEEDS = {"S": 112.0, "S*": 92.0}         # engineers leave with the unit / engineers kept
TECH_SHARE, W_LOST = 0.10, 5.0              # the unit takes 10% of technology; 5 capability points if the team goes
BUILD, PER_M = 0.5, 0.05                    # the build commits half the liquidity; 0.05 points per $1 million
PRE_D = FLOOR / (1 - BUILD)                 # 700: the floor still holds after the build
TARGET_T = 50.0                             # the technology target of Figure 6.3

class Operator:
    # Definition 6.1: a declared action. Its domain holds the preconditions (Definition 6.3): `needs` lists
    # (test, reason) pairs. Its effect is the mapping (Definition 6.2). Outside the domain there is no result.
    def __init__(self, code, name, needs, effect, domain):
        self.code, self.name, self.needs, self.effect, self.domain = code, name, needs, effect, domain
    def why_not(self, x):
        for test, reason in self.needs:
            if not test(x):
                return reason(x)
        return None
    def __call__(self, x):
        return self.effect(x) if self.why_not(x) is None else None

def sale(keep):
    return lambda x: x._replace(L=x.L + PROCEEDS["S*" if keep else "S"], T=(1 - TECH_SHARE) * x.T,
                                W=x.W if keep else x.W - W_LOST, sold=True)

UNSOLD = (lambda x: not x.sold, lambda x: "the unit is already sold")
STAFFED = (lambda x: x.W >= W_MIN, lambda x: f"workforce {x.W:.0f} < {W_MIN:.0f}")
S = Operator("S", "sell, engineers leave", [UNSOLD], sale(False), "the unit unsold")
S_ = Operator("S*", "sell, engineers kept", [UNSOLD], sale(True), "the unit unsold")
D = Operator("D", "digital build",
             [STAFFED, (lambda x: (1 - BUILD) * x.L >= FLOOR, lambda x: f"leaves ${(1 - BUILD) * x.L:.0f}m, below the floor")],
             lambda x: x._replace(L=(1 - BUILD) * x.L, T=x.T + PER_M * BUILD * x.L),
             f"workforce >= {W_MIN:.0f}, liquidity >= ${PRE_D:.0f}m")
A = Operator("A", "automation round",
             [STAFFED, (lambda x: x.L >= PRE_A, lambda x: f"liquidity ${x.L:.0f}m < ${PRE_A:.0f}m")],
             lambda x: x._replace(L=x.L - COST, W=x.W + W_GAIN, E=x.E + CLOSE * (CEIL - x.E)),
             f"workforce >= {W_MIN:.0f}, liquidity >= ${PRE_A:.0f}m")
REPERTOIRE = [S, S_, D, A]

def show(x):
    return f"${x.L:.0f}m, T {x.T:.1f}, W {x.W:.0f}"

def half_up(v, nd):
    # the book rounds halves up (84.125 -> 84.13); Python's own formatting rounds them to even
    return float(Decimal(repr(float(v))).quantize(Decimal(1).scaleb(-nd), rounding=ROUND_HALF_UP))

print("Meridian today:", show(TODAY) + f", E {TODAY.E:.0f}; the board's floor ${FLOOR:.0f}m")
print(f"automation round needs ${PRE_A:.0f}m = floor ${FLOOR:.0f}m + cost ${COST:.0f}m;"
      f" digital build needs ${PRE_D:.0f}m, since it keeps only half")

## Panel 1 — Forge the operators (Definitions 6.1–6.4; Figure 6.1; Proposition 6.1; Theorem 6.1)

AXIOM Lab 1.6, step 1: declare the four moves with their effects and preconditions, the board's floor included, and
inspect each operator's domain and range. From today's state three of the four moves can be taken. The digital build
cannot: it would leave \$310 million, below the floor, and outside its domain an operator has no result ("cannot",
not "should not"). One automation round takes Meridian from \$620 million and efficiency 71 to **\$500 million and
efficiency 78** (Figure 6.1). Two more readings of the declarations: a record that pairs one starting state with two
outcomes is not one operator (Proposition 6.1), which is why the sale is a family of two, one for each decision about
the engineers; and the sale's proceeds rest on a buyer paying the assumed price, an assumption about the environment
rather than a commitment (Theorem 6.1(ii)).

In [ ]:
print(f"{'move':26s} {'domain (preconditions)':37s} from today")
first = {}
for op in REPERTOIRE:
    y = op(TODAY)
    first[op.code] = y
    res = (show(y) + (f", E {y.E:.0f}" if op is A else "")) if y else "no result: " + op.why_not(TODAY)
    print(f"{op.code:3s}{op.name:23s} {op.domain:37s} {res}")
n_first = sum(y is not None for y in first.values())
print(f"{n_first} of the 4 moves can be taken today")
one = first["A"]

# Figure 6.1: the automation round on the (liquidity, efficiency) plane, workforce capability 58
print("\nthe round on Figure 6.1's plane, (liquidity $m, efficiency) before -> after:")
for q, e, note in ((620.0, 71.0, "Meridian today"), (860.0, 67.5, ""), (800.0, 81.0, ""), (410.0, 66.5, "")):
    x = TODAY._replace(L=q, E=e)
    y = A(x)
    print(f"  ({q:.0f}, {e:4.1f}) -> " + (f"({y.L:.0f}, {y.E:5.2f})" if y else "no result: " + A.why_not(x)
                                        + ", a feasible state outside the domain") + (f"   {note}" if note else ""))
print("range: liquidity at least the floor; efficiency (e + 85)/2, below 85 whenever e is: never the ceiling itself")

# Proposition 6.1: a record of (before, after) pairs is one operator exactly when no start has two outcomes
record = [(TODAY, S(TODAY)), (TODAY, S_(TODAY))]
outcomes = {}
for before, after in record:
    outcomes.setdefault(before, set()).add(after)
functional = all(len(v) == 1 for v in outcomes.values())
print(f"\nrecord of 'sell part of Industrial Systems' from today: {len(outcomes[TODAY])} outcomes for one start,"
      f" {'functional' if functional else 'not functional'} (Proposition 6.1)")
print("  so the sale is a family of two operators, S and S*, one for each decision about the engineers (Definition 6.1)")
print("the sale's proceeds rest on a buyer paying the assumed price: an assumption, not a commitment (Theorem 6.1(ii))")

## Panel 2 — A policy turns a family into a system (Theorem 6.1(iii); Online Appendix Example 6.B.1)

A bank invests in its digital platform from the liquidity it holds above a regulatory floor of \$300 million, and each
\$1 million invested adds 0.05 points to its technology index. The amount $u$ is a decision, so the program is a family
$T_u(q, t) = (q - u,\ t + 0.05u)$ with domain $q - u \ge 300$. The board's rule, invest half of the liquidity above the
floor each year, $u(q) = \tfrac12(q - 300)$, selects one operator at every state and leaves a single closed-loop map,
$q' = \tfrac12(q + 300)$: a contraction with $L = 1/2$ toward the floor. From \$900 million, liquidity is **\$600,
\$450, \$375 and \$337.5 million**, approaching but never reaching \$300 million. The investments sum to the initial
slack of \$600 million, so technology rises from 41 toward **71** (56 after one year, 63.5 after two). A board that
wants faster progress must change the rule, not hope for a different path.

In [ ]:
BANK_Q0, BANK_FLOOR, BANK_T0, GAIN = 900.0, 300.0, 41.0, 0.05

def invest(u):
    # one member of the family: T_u(q, t) = (q - u, t + 0.05 u) on the domain q - u >= 300
    return lambda q, t: (q - u, t + GAIN * u) if q - u >= BANK_FLOOR - 1e-12 else None

def rule(q):
    return 0.5 * (q - BANK_FLOOR)      # the board's policy: half of the liquidity above the floor

q, t, bank = BANK_Q0, BANK_T0, [(0, BANK_Q0, 0.0, BANK_T0)]
for year in range(1, 41):
    u = rule(q)
    q, t = invest(u)(q, t)
    bank.append((year, q, u, t))
print("year  liquidity  invested  technology   300 + 600 x 0.5^k")
for yr, q, u, t in bank[:7]:
    print(f"{yr:4d} {q:10.2f} {u:9.2f} {t:11.2f} {BANK_FLOOR + (BANK_Q0 - BANK_FLOOR) * 0.5 ** yr:16.2f}")
invested = sum(r[2] for r in bank)
print(f"closed loop q' = (q + 300)/2, a contraction with L = 0.5: after 40 years ${bank[-1][1]:.9f}m, still above $300m")
print(f"investments sum to ${invested:.6f}m, the initial slack of $600m; technology -> {bank[-1][3]:.6f}"
      f" (41 + 0.05 x 600 = 71)")

## Panel 3 — Compose programs (Theorem 6.2(iii); Proposition 6.5; Figure 6.3)

AXIOM Lab 1.6, step 2: chain the moves two at a time. A program runs only if each step's result lies in the next
step's domain (Definition 6.5), so every seam is a membership check. Selling as designed is a dead end: with workforce
capability down to 53, neither the digital build nor automation can follow. **Exactly one** two-move program reaches a
technology index of at least 50: sell while keeping the engineers, then build, which ends with **\$356 million and
technology 54.7**, only \$6 million above the floor. Order decides whether it runs at all (Theorem 6.2(iii)): the sale
is an *enabling* step that lifts liquidity to where the build respects the floor (Online Appendix 6.B). Section 6.5's
case on file: Worked Example 3.1's digital-then-restructure sequence ends with \$422 million but passes through
\$310 million, below the floor, so it is not feasible, whatever its end point (Proposition 6.5).

In [ ]:
pairs = {}
print("today:", show(TODAY))
for f in REPERTOIRE:
    y = f(TODAY)
    if y is None:
        print(f"{f.code}, {f.name}: not possible ({f.why_not(TODAY)})")
        pairs.update({(f.code, g.code): None for g in REPERTOIRE})
        continue
    print(f"{f.code}, {f.name} -> {show(y)}")
    for g in REPERTOIRE:
        z = g(y)
        pairs[(f.code, g.code)] = z
        if g in (S, S_) and y.sold:
            continue                                   # a second sale is impossible and is not drawn (Figure 6.3)
        mark = f"   <- target: technology >= {TARGET_T:.0f}, ${z.L - FLOOR:.0f}m above the floor" if z and z.T >= TARGET_T else ""
        print(f"    {f.code}, then {g.code}: " + (show(z) + mark if z else f"not possible ({g.why_not(y)})"))
ran = [p for p, z in pairs.items() if z is not None]
hits = [p for p in ran if pairs[p].T >= TARGET_T and pairs[p].L >= FLOOR]
win = pairs[hits[0]]
print(f"\n{len(ran)} of the 16 ordered pairs pass every seam; {len(hits)} reaches the target: {hits[0][0]} then {hits[0][1]}")
print(f"order decides whether it runs (Theorem 6.2(iii)): {hits[0][0]} then {hits[0][1]} gives {show(win)};"
      f" {hits[0][1]} then {hits[0][0]} has no result today")
print(f"{hits[0][0]} is an enabling step: it lifts liquidity from ${TODAY.L:.0f}m to ${first['S*'].L:.0f}m,"
      f" above the ${PRE_D:.0f}m at which the build respects the floor")

# Section 6.5: Worked Example 3.1's digital-then-restructure, with the build's floor precondition switched off
D_unchecked = Operator("D", "digital build, floor ignored", [STAFFED], D.effect, "workforce >= 55")
mid = D_unchecked(TODAY)
end = S(mid)
print(f"\ndigital, then restructure (Worked Example 3.1): ${TODAY.L:.0f}m -> ${mid.L:.0f}m -> ${end.L:.0f}m,"
      f" technology {half_up(end.T, 1):.1f}, workforce {end.W:.0f}")
print(f"  it ends ${end.L - FLOOR:.0f}m above the floor but passes ${FLOOR - mid.L:.0f}m below it: not feasible (Proposition 6.5)")

## Panel 4 — Test the playbook: savings paid out (Worked Example 6.1, Step 2; Equation 6.2; Theorem 6.3)

AXIOM Lab 1.6, step 3, first part. Repeating the round is iterating the operator, and each further round adds a
precondition on today's state: $\operatorname{dom}T^{k+1} = \{x \in \operatorname{dom}T^{k} : T^{k}x \in
\operatorname{dom}T\}$ (Equation 6.2). If the savings leave Meridian, to fund dividends say, each round simply spends
\$120 million: liquidity goes from \$620 million to **\$500 million** and then to **\$380 million**, and the third
round's precondition fails. The $k$-th round requires today's liquidity of at least $470 + 120(k - 1)$ million, so the
nested domains shrink to nothing and the **sustainable core is empty** (Equation 6.3; Theorem 6.3): on these terms
automation is a trade. The workforce precondition is not the problem; each round renews it.

In [ ]:
def playbook(q0=TODAY.L, e0=TODAY.E, w0=TODAY.W, recycle=1.0, shortfall=0.0, years=40, check=True):
    # One automation round at the start of each year (Worked Example 6.1): the precondition is checked before the
    # round; its cost and the year's savings ($10m a year per point above today's 71, the share `recycle` kept)
    # land by the year's end. Returns rows (year, efficiency, savings, savings less the cost, liquidity at year
    # end) and the first round whose precondition fails (None if every round runs).
    rows, q, e, w = [(0, e0, 0.0, 0.0, q0)], q0, e0, w0
    for k in range(1, years + 1):
        if check and not (q >= PRE_A - 1e-9 and w >= W_MIN):
            return rows, k
        e = e + CLOSE * (CEIL - e) - shortfall
        w += W_GAIN
        sav = recycle * VALUE * (e - TODAY.E)
        q = q - COST + sav
        rows.append((k, e, sav, sav - COST, q))
    return rows, None

def thresholds(n=10, **kw):
    # Equation 6.2 in today's liquidity: round k runs from q0 exactly when q0 - drop_j >= 470 for every j < k, where
    # drop_j is how far liquidity has fallen below its start by the end of year j (the same amount from any start)
    rows, _ = playbook(check=False, years=n, **kw)
    drop = [TODAY.L - r[4] for r in rows]
    return [PRE_A + max(drop[:k]) for k in range(1, n + 1)]

paid, stop_paid = playbook(recycle=0.0)
print("savings paid out: liquidity " + " -> ".join(f"${r[4]:.0f}m" for r in paid)
      + f"; round {stop_paid} not available (${paid[-1][4]:.0f}m < ${PRE_A:.0f}m)")
print(f"rounds that run: {len(paid) - 1}; workforce capability after them {TODAY.W + (len(paid) - 1) * W_GAIN:.0f}"
      " (each round renews that precondition)")
th_paid = thresholds(recycle=0.0)
print("\nnested domains (Equation 6.2): the liquidity today that lets the playbook run k rounds")
print("  round k  " + "".join(f"{k:7d}" for k in range(1, 11)))
print("  paid out " + "".join(f"{v:7.0f}" for v in th_paid))
print(f"  = 470 + 120(k - 1): it rises ${th_paid[1] - th_paid[0]:.0f}m a round without bound, so no start supports every"
      " round: the sustainable core is empty (Theorem 6.3)")

## Panel 5 — Test the playbook: savings recycled (Worked Example 6.1, Step 3; Table 6.4; Theorem 6.3)

AXIOM Lab 1.6, step 3, second part. Keep the savings in Meridian's liquidity, so that each year's composite operator
is "run a round, then bank the year's savings". The savings grow from \$70 million to **\$122.5 million in year 3**,
when they first exceed the round's cost; liquidity bottoms at **\$555 million** at the end of year 2 and then rises by
up to \$20 million a year (Table 6.4). The tightest moment is the start of year 3, so the nested domains stop shrinking:
from today's efficiency the playbook runs indefinitely from any starting liquidity of **\$535 million** or more, and
Meridian, at \$620 million, is inside the sustainable core with **\$85 million** to spare. The lab asks for that
threshold; the cell finds it from the nested domains and by bisection, and checks that the core is self-sustaining:
one more round from a state in the core lands in the core (Theorem 6.3(ii)).

In [ ]:
rec, stop_rec = playbook()
row = lambda name, vals: print(f"  {name:31s}" + "".join(f"{v:>9s}" for v in vals))
print("Table 6.4: the automation playbook with the savings recycled, as designed ($ million; one round at the start of each year)")
row("Year", [str(r[0]) for r in rec[:7]])
row("Efficiency (index)", [f"{half_up(r[1], 2):.2f}" for r in rec[:7]])
row("Savings this year", ["--"] + [f"{half_up(r[2], 1):.1f}" for r in rec[1:7]])
row("Savings less the round's cost", ["--"] + [f"{half_up(r[3], 1):.1f}" for r in rec[1:7]])
row("Liquidity at year end", [f"{half_up(r[4], 1):.1f}" for r in rec[:7]])
self_year = next(r[0] for r in rec[1:] if r[2] > COST)
low = min(rec[1:], key=lambda r: r[4])
print(f"savings first exceed the round's cost in year {self_year}; liquidity bottoms at ${low[4]:.0f}m at the end of year"
      f" {low[0]}; in year 40 it rises ${rec[-1][3]:.2f}m (the limit: savings of ${VALUE * (CEIL - TODAY.E):.0f}m less the"
      f" round's ${COST:.0f}m)")

th_rec = thresholds()
print("\nnested domains: the liquidity today that lets the playbook run k rounds")
print("  round k  " + "".join(f"{k:7d}" for k in range(1, 11)))
print("  recycled " + "".join(f"{v:7.1f}" for v in th_rec))
core = max(thresholds(n=200))
lo, hi = 300.0, 900.0
for _ in range(60):                                   # bisection on today's liquidity, every round for 200 years
    q_mid = (lo + hi) / 2
    lo, hi = (lo, q_mid) if playbook(q0=q_mid, years=200)[1] is None else (q_mid, hi)
print(f"the domains settle: the core starts at ${core:.2f}m (bisection over 200 years: ${hi:.2f}m);"
      f" Meridian, at ${TODAY.L:.0f}m, is inside with ${TODAY.L - core:.0f}m to spare")

def core_at(e, n=200):
    # the core's threshold for a state with efficiency e: from q0 = 0 the playbook's path is minus its drop
    rows, _ = playbook(q0=0.0, e0=e, check=False, years=n)
    return PRE_A + max(-r[4] for r in rows)

print("\nTheorem 6.3(ii): each year's state lies in the core of the playbook that remains")
print("  year  liquidity  efficiency  core threshold  margin")
for r in rec[:7]:
    c = core_at(r[1])
    print(f"  {r[0]:4d} {r[4]:10.2f} {r[1]:11.3f} {c:15.2f} {r[4] - c:7.2f}")

k = np.arange(1, 11)
fig, ax = plt.subplots(figsize=(7.6, 3.6))
ax.plot(k, th_paid, color="#616161", lw=1.8, ls=":", marker="o", ms=4, label="savings paid out: 470 + 120(k - 1), no core")
ax.plot(k, th_rec, color="#1F4E79", lw=2, marker="o", ms=4, label=f"savings recycled: settles at {core:.0f}, the core")
ax.axhline(TODAY.L, color="#E65100", lw=1, ls="--")
ax.text(10.1, TODAY.L + 6, "Meridian today: \\$620m", ha="right", va="bottom", fontsize=8, color="#E65100")
ax.annotate("", xy=(6, core), xytext=(6, TODAY.L), arrowprops=dict(arrowstyle="<->", color="#1F4E79", lw=0.8,
            shrinkA=0, shrinkB=0))
ax.text(6.12, (core + TODAY.L) / 2, f"Meridian is \\${TODAY.L - core:.0f}m inside the core", va="center", fontsize=8,
        color="#1F4E79")
ax.set_xticks(k)
ax.set(xlim=(0.6, 10.4), ylim=(450, 800), xlabel="round k (one a year)", ylabel="liquidity today that lets k rounds run ($ million)",
       title="Recycled savings stop the nested domains from shrinking (seed 26106)")
ax.legend(frameon=False, fontsize=8, loc="upper left"); plt.tight_layout(); plt.show()

## Panel 6 — Explore reachability (Theorem 6.4; Online Appendix Remark 6.B.3)

AXIOM Lab 1.6, step 4: grow the reachable sets of the four-move repertoire one step at a time by breadth-first search,
$R_{k+1} = R_k \cup \bigcup_i T_i(R_k \cap \operatorname{dom}T_i)$, against the technology target. Attainability is a
membership question (Theorem 6.4(i)): the target is first met at depth two, by the one path of Panel 3. The counts by
depth are computed from the book's declarations: the sets stop growing after four moves, so the repertoire reaches
finitely many states and no program from today has more than four moves. Then tighten or loosen the target, and its
cushion over the floor. Theorem 6.4(iii) explains why complete goals need tolerances: efficiency after $k$ recycled
rounds is $85 - 14 \cdot 0.5^{k}$ (the first row of Table 6.4), so the ceiling of 85 is approached but never reached.
Within a tolerance $\varepsilon$ it is reached after $\lceil \log_2(14/\varepsilon) \rceil$ rounds, and halving the
tolerance costs one more round (Remark 6.B.3).

In [ ]:
def key(x):
    return (round(x.L, 6), round(x.T, 6), x.W, round(x.E, 6), x.sold)

def reach(x0=TODAY, repertoire=REPERTOIRE, kmax=12):
    # Theorem 6.4(i): breadth-first search; each state keeps the first, shortest, word that reaches it
    seen, frontier, sizes = {key(x0): (x0, (), 0)}, [(x0, ())], [1]
    for k in range(1, kmax + 1):
        nxt = []
        for x, word in frontier:
            for op in repertoire:
                y = op(x)
                if y is not None and key(y) not in seen:
                    seen[key(y)] = (y, word + (op.code,), k)
                    nxt.append((y, word + (op.code,)))
        sizes.append(len(seen))
        frontier = nxt
        if not nxt:
            break
    return seen, sizes

def programs(x0=TODAY, repertoire=REPERTOIRE, kmax=12):
    # every word that runs from x0, counted by length (depth first)
    count, stack = {}, [(x0, 0)]
    while stack:
        x, k = stack.pop()
        for op in repertoire:
            y = op(x)
            if y is not None and k < kmax:
                count[k + 1] = count.get(k + 1, 0) + 1
                stack.append((y, k + 1))
    return count

seen, sizes = reach()
nprog = programs()
closed = next(k for k in range(1, len(sizes)) if sizes[k] == sizes[k - 1]) - 1
print("depth k                 " + "".join(f"{k:5d}" for k in range(len(sizes))))
print("states in R_k           " + "".join(f"{s:5d}" for s in sizes))
print("programs of length k    " + "".join(f"{nprog.get(k, 1 if k == 0 else 0):5d}" for k in range(len(sizes))))
print(f"R_k stops growing after depth {closed}: {sizes[-1]} reachable states; the longest program has {max(nprog)} moves")
for x, word, d in sorted(seen.values(), key=lambda v: (v[2], -v[0].L)):
    tag = "   meets the target" if x.T >= TARGET_T else ""
    print(f"  depth {d}: {' '.join(word) or 'today':9s} {show(x)}, E {x.E:5.2f}{tag}")

def first_hit(tau, cushion=0.0):
    # the shallowest reachable state with technology >= tau and liquidity >= floor + cushion (None if unreachable)
    ok = [(d, word, x) for x, word, d in seen.values() if x.T >= tau - 1e-9 and x.L >= FLOOR + cushion - 1e-9]
    return min(ok, key=lambda h: (h[0], -h[2].L)) if ok else None

levels = sorted({round(x.T, 1) for x, w, d in seen.values()})
print(f"\ntechnology levels the repertoire can reach: {levels}")
for tau, cushion in ((41.0, 0.0), (45.0, 0.0), (50.0, 0.0), (54.7, 0.0), (55.0, 0.0), (50.0, 6.0), (50.0, 7.0)):
    h = first_hit(tau, cushion)
    res = f"depth {h[0]}, {' '.join(h[1]) or 'today'}: {show(h[2])}" if h else "not reachable"
    print(f"  technology >= {tau:4.1f}, liquidity >= ${FLOOR + cushion:.0f}m: {res}")
target = first_hit(TARGET_T)

# Theorem 6.4(iii): a complete goal. Efficiency after k recycled rounds is 85 - 14 x 0.5^k: never 85 itself
gap0, L_eff = CEIL - TODAY.E, 1 - CLOSE
assert all(abs(r[1] - (CEIL - gap0 * L_eff ** r[0])) < 1e-9 for r in rec)
print(f"\nefficiency after k rounds: 85 - {gap0:.0f} x 0.5^k; within a tolerance eps of 85 after ceil(log2({gap0:.0f}/eps)) rounds"
      " (Remark 6.B.3)")
tol_rounds = {}
for eps in (2.0, 1.0, 0.5, 0.25, 0.1):
    n = math.ceil(math.log(gap0 / eps) / math.log(1 / L_eff))
    tol_rounds[eps] = n
    print(f"  eps = {eps:4.2f}: {n} rounds (efficiency {CEIL - gap0 * L_eff ** n:.4f}; one fewer leaves"
          f" {CEIL - gap0 * L_eff ** (n - 1):.4f})")
print(f"halving the tolerance costs log 2 / log(1/L) = {math.log(2) / math.log(1 / L_eff):.0f} more round at L = 0.5")
print("Theorem 6.4(iii)'s own example, T x = x/2 from 1: the target {0} is never reached; [0, eps] after ceil(log2(1/eps)) steps:",
      ", ".join(f"eps {e}: {math.ceil(math.log2(1 / e))}" for e in (0.1, 0.01, 0.001)))

fig, ax = plt.subplots(figsize=(7.6, 3.6))
ax.axvspan(180, FLOOR, color="#F6E3E3", lw=0)
ax.axvline(FLOOR, color="#B71C1C", lw=0.9, ls="--")
ax.text(186, -0.42, "below the floor:\nno result", ha="left", va="top", fontsize=7.5, color="#B71C1C", linespacing=1.0)
depth_of = {key(x): d for x, w, d in seen.values()}
for x, word, d in seen.values():                      # every move that runs, from layer d to layer d + 1
    for op in REPERTOIRE:
        y = op(x)
        if y is not None:
            ax.annotate("", xy=(y.L, depth_of[key(y)]), xytext=(x.L, d),
                        arrowprops=dict(arrowstyle="-|>", color={"D": "#E65100", "A": "#1F4E79"}.get(op.code, "#999999"),
                                        lw=0.8, mutation_scale=7, shrinkA=5, shrinkB=5))
for start, d0, word in ((TODAY, 0, "D"), (one, 1, "A D")):   # the build would breach the floor: no result
    y = D_unchecked(start)
    ax.annotate("", xy=(y.L, d0 + 1), xytext=(start.L, d0), arrowprops=dict(arrowstyle="-|>", color="#B71C1C", lw=0.8,
                ls=(0, (3, 2)), mutation_scale=7, shrinkA=5, shrinkB=5))
    ax.plot(y.L, d0 + 1, "x", color="#B71C1C", ms=6, mew=1.5)
    ax.text(y.L - 9, d0 + 1, f"{word}: \\${y.L:.0f}m", ha="right", va="center", fontsize=7.5, color="#B71C1C")
tone = lambda x: "#E65100" if x.T >= TARGET_T else ("#1F4E79" if x.T > 40 else "#616161")
PLACE = {"": (12, 0, "left", "center"), "S": (9, 0, "left", "center"), "S*": (0, 0.3, "center", "top"),
         "A": (-7, -0.2, "right", "bottom"), "S* D": (-10, 0.28, "right", "top"), "A A": (0, 0.3, "center", "top"),
         "S* A": (0, 0.3, "center", "top"), "A S": (9, 0, "left", "center"), "S* A A": (-9, 0, "right", "center"),
         "A A S": (9, 0, "left", "center"), "S* A A A": (-9, 0, "right", "center"), "A A S A": (9, 0, "left", "center")}
for x, word, d in seen.values():
    w = " ".join(word)
    hit = x.T >= TARGET_T
    ax.plot(x.L, d, "*" if hit else "o", color=tone(x), ms=14 if hit else (8 if d == 0 else 5.5), zorder=5)
    dx, dy, ha, va = PLACE.get(w, (9, 0, "left", "center"))
    text = {"": "today: \\$620m, technology 41", "S* D": "S* D: \\$356m, technology 54.7"}.get(w, w)
    ax.text(x.L + dx, d + dy, text, ha=ha, va=va, fontsize=7.5, color=tone(x), fontweight="bold" if hit else "normal")
ax.plot([], [], "o", color="#1F4E79", ms=5, label="technology 41")
ax.plot([], [], "o", color="#616161", ms=5, label="technology 36.9")
ax.plot([], [], "*", color="#E65100", ms=9, label="technology 54.7: target")
ax.plot([], [], color="#999999", lw=1, label="sale, S or S*")
ax.plot([], [], color="#E65100", lw=1, label="digital build, D")
ax.plot([], [], color="#1F4E79", lw=1, label="automation round, A")
ax.set_yticks(range(closed + 1))
ax.set(xlim=(180, 800), ylim=(closed + 0.5, -0.6), xlabel="liquidity ($ million)", ylabel="moves from today",
       title="Only one reachable state meets the technology target (seed 26106)")
ax.legend(frameon=False, fontsize=7, loc="lower right", ncol=2, columnspacing=1.2, handlelength=1.6)
plt.tight_layout(); plt.show()

## Panel 7 — Stress the execution: the playbook (Worked Example 6.1, Step 4; Theorem 6.5(iii); Figure 6.2)

Suppose every round delivers one efficiency point less than designed. The efficiency step is a contraction with
$L = 0.5$ (each round halves the remaining gap), so by Theorem 6.5(iii) its steady state moves by at most
$1/(1 - 0.5) = 2$ points: efficiency settles at **83** instead of 85. Savings then top out at **\$120 million**,
exactly the cost of a round, and liquidity sinks toward **\$500 million** but never below the \$470 million
precondition: the playbook survives, with only **\$30 million** of headroom. With a two-point shortfall, savings top
out at **\$100 million**, liquidity falls below \$470 million before the fifth round, and the playbook ends: a trade
again. Whether automation is a strategy or a trade is set by two choices the board controls: what happens to the
savings, and how closely execution tracks the design.

In [ ]:
cases = {"savings paid out": dict(recycle=0.0), "recycled, as designed": dict(),
         "recycled, 1-point shortfall": dict(shortfall=1.0), "recycled, 2-point shortfall": dict(shortfall=2.0)}
runs = {name: playbook(**kw) for name, kw in cases.items()}
steady = {}
for s in (1.0, 2.0):
    e_star = CEIL - s / (1 - L_eff)                   # the fixed point of e' = e + 0.5 (85 - e) - s
    steady[s] = (e_star, VALUE * (e_star - TODAY.E))
    print(f"{s:.0f}-point shortfall a round: efficiency settles at {e_star:.0f}, moved {CEIL - e_star:.0f} = s/(1 - L)"
          f" (Theorem 6.5(iii)); savings top out at ${steady[s][1]:.0f}m against the round's ${COST:.0f}m")
r1, stop1 = runs["recycled, 1-point shortfall"]
long1, _ = playbook(shortfall=1.0, years=200)
print(f"1 point: liquidity " + " -> ".join(f"{r[4]:.2f}" for r in r1[1:7]) + f" ... -> ${long1[-1][4]:.2f}m:"
      f" never below ${PRE_A:.0f}m, ${long1[-1][4] - PRE_A:.0f}m of headroom; every round runs: {stop1 is None}")
r2, stop2 = runs["recycled, 2-point shortfall"]
print(f"2 points: liquidity " + " -> ".join(f"{r[4]:.2f}" for r in r2[1:]) + f": round {stop2} not available; a trade again")
th1, th2 = thresholds(n=200, shortfall=1.0), thresholds(n=10, shortfall=2.0)
print(f"nested domains: with 1 point the core starts at ${max(th1):.2f}m (Meridian ${TODAY.L - max(th1):.0f}m inside);"
      f" with 2 points round k needs " + ", ".join(f"{v:.2f}" for v in th2[:6]) + " ...: rising without bound, no core")

# the liquidity account keeps every shortfall: each year's savings fall 10 x the efficiency gap short of the design
design, _ = playbook(check=False, years=10)
short1, _ = playbook(shortfall=1.0, check=False, years=10)
gaps = [a[4] - b[4] for a, b in zip(design, short1)]
print("gap between designed and executed liquidity, 1 point, years 1-10: " + ", ".join(f"{g:.1f}" for g in gaps[1:]))
print(f"  growing by close to $20m a year: efficiency forgives the error (L = 0.5), the liquidity account keeps it")

fig, ax = plt.subplots(figsize=(7.6, 3.6))
style = {"savings paid out": ("#616161", ":"), "recycled, as designed": ("#1F4E79", "-"),
         "recycled, 1-point shortfall": ("#E65100", "--"), "recycled, 2-point shortfall": ("#B71C1C", "-.")}
for name, (rows, stop) in runs.items():
    yrs = [r[0] for r in rows if r[0] <= 10]
    liq = [r[4] for r in rows if r[0] <= 10]
    col, ls = style[name]
    ax.plot(yrs, liq, color=col, ls=ls, lw=1.8, marker="o", ms=3.5,
            label=name + (f": stops before round {stop}" if stop else ": runs indefinitely"))
    if stop:
        ax.plot(yrs[-1], liq[-1], "X", color=col, ms=9, zorder=6)
ax.axhline(PRE_A, color="#1F4E79", lw=0.9)
ax.text(10.2, PRE_A + 4, "precondition for a round: \\$470m", ha="right", va="bottom", fontsize=8, color="#1F4E79")
ax.axhline(FLOOR, color="#B71C1C", lw=0.9, ls="--")
ax.text(10.2, FLOOR + 4, "board floor: \\$350m", ha="right", va="bottom", fontsize=8, color="#B71C1C")
ax.set_xticks(range(11))
ax.set(xlim=(-0.3, 10.4), ylim=(330, 700), xlabel="year (one automation round at the start of each year)",
       ylabel="liquidity at year end ($ million)", title="Recycled savings make automation a strategy (seed 26106)")
ax.legend(frameon=False, fontsize=7.5, loc="upper left", ncol=2); plt.tight_layout(); plt.show()

## Panel 8 — Stress the execution: drift and its bound (Theorem 6.5(i)–(ii); Figure 6.4; Online Appendix Remark 6.B.2)

AXIOM Lab 1.6, step 5: set the shortfall per step $\varepsilon$ and the sensitivity factor $L$, and compare the observed
drift with the bound of Equation 6.4, $\varepsilon(1 + L + \cdots + L^{k-1})$, never more than $\varepsilon/(1 - L)$
under contraction. With a 1 percent shortfall each quarter, a consolidating program ($L = 0.8$) never drifts more than
**5 percent**, a neutral one drifts in proportion to time, **12 percent** in three years, and an amplifying one
($L = 1.2$) drifts **about 40 percent** in three years. The bound is attained by $Tx = Lx + c$ executed as
$Lx + c - \varepsilon$ (Remark 6.B.2), so Figure 6.4's curves are the worst case itself. The lab's new draws
(seed 26106) let each quarter's shortfall vary between zero and 1 percent: every observed path stays inside the
bound. Hence the doctrine: match execution tolerance to the program's sensitivity factor.

In [ ]:
EPS, K = 1.0, 12                                   # 1 percent of plan a quarter, for three years
LS = (0.8, 1.0, 1.2)

def drift(L, k, eps=EPS):
    # Equation 6.4: eps (1 + L + ... + L^(k-1))
    return eps * k if L == 1 or k == 0 else eps * (L ** k - 1) / (L - 1)

quarters = np.arange(K + 1)
print("quarter " + "".join(f"{k:6d}" for k in quarters))
for L in LS:
    print(f"L = {L:.1f} " + "".join(f"{drift(L, k):6.2f}" for k in quarters))
print(f"under contraction the drift never exceeds eps/(1 - L) = {EPS / (1 - 0.8):.0f} percent at L = 0.8")

# Remark 6.B.2: the bound is attained by T x = L x + c executed as T' x = L x + c - eps (design held at 100)
attained = {}
for L in LS:
    x = xe = 100.0
    for _ in range(K):
        x, xe = L * x + 100 * (1 - L), L * xe + 100 * (1 - L) - EPS
    attained[L] = x - xe
    print(f"L = {L}: the executed path ends {x - xe:.4f} below the design; the bound is {drift(L, K):.4f}")
print("Theorem 6.5(i): an error of 1 in the starting state after 12 quarters is " + ", ".join(f"{L ** K:.3f} at L = {L}" for L in LS))
print(f"to hold the drift of a consolidating program, an amplifying one needs execution {drift(1.2, K) / drift(0.8, K):.1f} times as precise")

# the lab's new draws (seed 26106): each quarter's shortfall uniform between 0 and eps
rng = np.random.default_rng(SEED)
N = 2000
draws = rng.uniform(0.0, EPS, size=(len(LS), N, K))
paths = {}
for i, L in enumerate(LS):
    d = np.zeros((N, K + 1))
    for k in range(K):
        d[:, k + 1] = L * d[:, k] + draws[i, :, k]
    paths[L] = d
    inside = bool((d <= np.array([drift(L, k) for k in quarters]) + 1e-12).all())
    print(f"seed {SEED}, L = {L}: at quarter 12 mean {d[:, -1].mean():5.2f}, largest {d[:, -1].max():5.2f} of {N} paths;"
          f" bound {drift(L, K):5.2f}; every path inside: {inside}")

fig, ax = plt.subplots(figsize=(7.6, 3.6))
for L, col, name in ((1.2, "#B71C1C", "amplifying, L = 1.2"), (1.0, "#616161", "neutral, L = 1.0"),
                     (0.8, "#1F4E79", "consolidating, L = 0.8")):
    for j in range(40):
        ax.plot(quarters, paths[L][j], color=col, lw=0.5, alpha=0.25)
    ax.plot(quarters, [drift(L, k) for k in quarters], color=col, lw=2, marker="o", ms=3.5, label=f"{name}: worst case")
    ax.text(12.25, drift(L, K), f"{drift(L, K):.1f}%", va="center", fontsize=8, color=col)
ax.axhline(EPS / (1 - 0.8), color="#1F4E79", lw=0.8, ls="--")
ax.text(0.25, 5.5, "5% bound at L = 0.8", fontsize=8, color="#1F4E79")
ax.plot([], [], color="#999999", lw=0.8, label="shortfalls drawn between 0 and 1% a quarter")
ax.set_xticks(range(0, 13, 2))
ax.set(xlim=(0, 13.4), ylim=(0, 42), xlabel="quarters of execution", ylabel="gap from the designed path (% of plan)",
       title="Amplifying programs compound execution error (seed 26106)")
ax.legend(frameon=False, fontsize=8, loc="upper left"); plt.tight_layout(); plt.show()

## Validation — the book's numbers, and agreement with `DCT_V1_Ch06_Lab.xlsx`

Each checkpoint is a number printed in Chapter 6 (Figures 6.1–6.4, Sections 6.2 and 6.5–6.7, Worked Example 6.1 and
Table 6.4), in Worked Example 3.1, which Section 6.5 cites, or in Online Appendix 6.B. Values the book prints rounded
are checked to half a unit of their last digit (the book rounds halves up). The workbook recomputes the ones marked
*workbook* with Excel formulas and shows PASS against the same engine values on its `Checks` sheet.

In [ ]:
T64 = {r[0]: r for r in rec}
checks = [
    ("precondition for a round, liquidity (workbook)", PRE_A, 470, 0),
    ("moves possible from today (workbook)", n_first, 3, 0),
    ("one round from today: liquidity (workbook)", one.L, 500, 0),
    ("one round from today: efficiency (workbook)", one.E, 78, 0),
    ("digital build first: liquidity left (workbook)", (1 - BUILD) * TODAY.L, 310, 0),
    ("sell, engineers leave: liquidity (workbook)", first["S"].L, 732, 0),
    ("sell, engineers leave: technology (workbook)", first["S"].T, 36.9, 1e-9),
    ("sell, engineers leave: workforce (workbook)", first["S"].W, 53, 0),
    ("sell, engineers kept: liquidity (workbook)", first["S*"].L, 712, 0),
    ("two-move programs reaching the target (workbook)", len(hits), 1, 0),
    ("S* then D: liquidity (workbook)", win.L, 356, 0),
    ("S* then D: technology (workbook)", win.T, 54.7, 1e-9),
    ("S* then D: headroom over the floor (workbook)", win.L - FLOOR, 6, 0),
    ("A then S: liquidity (workbook)", pairs[("A", "S")].L, 612, 0),
    ("A then S: workforce (workbook)", pairs[("A", "S")].W, 54, 0),
    ("A then S*: liquidity (workbook)", pairs[("A", "S*")].L, 592, 0),
    ("A then A: liquidity (workbook)", pairs[("A", "A")].L, 380, 0),
    ("A then D: liquidity it would leave (workbook)", (1 - BUILD) * one.L, 250, 0),
    ("digital then restructure: end liquidity (workbook)", end.L, 422, 0),
    ("digital then restructure: technology (workbook)", end.T, 50.9, 0.05),
    ("paid out: liquidity after round 2 (workbook)", paid[2][4], 380, 0),
    ("paid out: rounds that run (workbook)", len(paid) - 1, 2, 0),
    ("paid out: extra liquidity each round needs (workbook)", th_paid[1] - th_paid[0], 120, 0),
] + [
    (f"Table 6.4: efficiency, year {y} (workbook)", T64[y][1], v, 0.005)
    for y, v in zip(range(1, 7), (78.00, 81.50, 83.25, 84.13, 84.56, 84.78))
] + [
    (f"Table 6.4: savings, year {y} (workbook)", T64[y][2], v, 0.05)
    for y, v in zip(range(1, 7), (70.0, 105.0, 122.5, 131.3, 135.6, 137.8))
] + [
    (f"Table 6.4: savings less cost, year {y} (workbook)", T64[y][3], v, 0.05)
    for y, v in zip(range(1, 7), (-50.0, -15.0, 2.5, 11.3, 15.6, 17.8))
] + [
    (f"Table 6.4: liquidity, year {y} (workbook)", T64[y][4], v, 0.05)
    for y, v in zip(range(1, 7), (570.0, 555.0, 557.5, 568.8, 584.4, 602.2))
] + [
    ("recycled: first year savings exceed the cost (workbook)", self_year, 3, 0),
    ("recycled: lowest liquidity (workbook)", low[4], 555, 0),
    ("recycled: long-run yearly rise of liquidity (workbook)", rec[-1][3], 20, 1e-6),
    ("core threshold at today's efficiency (workbook)", core, 535, 1e-9),
    ("core threshold by bisection", hi, 535, 1e-6),
    ("Meridian's margin inside the core (workbook)", TODAY.L - core, 85, 1e-9),
    ("1-point shortfall: steady efficiency (workbook)", steady[1.0][0], 83, 0),
    ("steady-state shift bound 1/(1 - 0.5) (workbook)", 1.0 / (1 - L_eff), 2, 0),
    ("1-point shortfall: savings top out (workbook)", steady[1.0][1], 120, 1e-9),
    ("1-point shortfall: liquidity tends to (workbook)", long1[-1][4], 500, 1e-6),
    ("1-point shortfall: headroom (workbook)", long1[-1][4] - PRE_A, 30, 1e-6),
    ("2-point shortfall: savings top out (workbook)", steady[2.0][1], 100, 1e-9),
    ("2-point shortfall: round that fails (workbook)", stop2, 5, 0),
    ("drift bound at L = 0.8, percent (workbook)", EPS / (1 - 0.8), 5, 1e-9),
    ("drift after 12 quarters, L = 0.8 (workbook)", drift(0.8, K), 4.7, 0.05),
    ("drift after 12 quarters, L = 1.0 (workbook)", drift(1.0, K), 12, 1e-9),
    ("drift after 12 quarters, L = 1.2 (workbook)", drift(1.2, K), 40, 0.5),
    ("bound less the worst case, L = 1.2 (workbook)", drift(1.2, K) - attained[1.2], 0, 1e-9),
    ("bank: liquidity after one year (workbook)", bank[1][1], 600, 1e-9),
    ("bank: liquidity after two years (workbook)", bank[2][1], 450, 1e-9),
    ("bank: liquidity after three years (workbook)", bank[3][1], 375, 1e-9),
    ("bank: liquidity after four years (workbook)", bank[4][1], 337.5, 1e-9),
    ("bank: technology after one year (workbook)", bank[1][3], 56, 1e-9),
    ("bank: technology after two years (workbook)", bank[2][3], 63.5, 1e-9),
    ("bank: total invested, the initial slack (workbook)", invested, 600, 1e-6),
    ("bank: technology in the limit (workbook)", bank[-1][3], 71, 1e-6),
]
ok, width = 0, max(len(name) for name, *_ in checks)
for name, got, want, tol in checks:
    status = "PASS" if abs(got - want) <= tol + 1e-9 else "FAIL"
    ok += status == "PASS"
    print(f"{name:{width}s} {got:10.4f}   book {want:9.4f}   {status}")
print(f"\n{ok} of {len(checks)} checkpoints PASS")

**Next.** Chapter 7 assembles the machine: the company as a process that unfolds in time under its inherited momentum,
management's decisions and a changing environment, with steady states, stability tests and management itself as a
feedback rule. Theorem 6.1(iii) is the bridge: a policy turns a family of operators into a closed-loop system.
AXIOM-06 runs the same model interactively (Lab 1.6); this notebook and the workbook are its reproducible record.